In [0]:
28# First, let's confirm you can access your S3 bucket
# Replace with your actual S3 path
s3_bucket_path = "workspace.fraud.tx_train_gold"  # Update this!

# # List files in S3 bucket
# display(dbutils.fs.ls(s3_bucket_path))

# If transactions.csv is in a subfolder, adjust the path
# Example: display(dbutils.fs.ls("s3://your-bucket-name/raw/"))


In [0]:
df = spark.table("workspace.fraud.tx_train_gold")

In [0]:
df.show()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# load transaction data

txn = spark.read.csv(s3_bucket_path + "transactions_data.csv", header=True, inferSchema=True)

# load cards data
# cards = spark.read.csv(s3_bucket_path + "cards_data.csv", header=True, inferSchema=True)

# load users data
# users = spark.read.csv(s3_bucket_path + "users_data.csv", header=True, inferSchema=True)

In [0]:
from pyspark.sql.functions import explode, col
from pyspark.sql.types import StructType, StructField, MapType, StringType

# # read from NDJSON file for labels which was generated in the previous notebook
output_path = s3_bucket_path + "fraud_labels_nd.json"

# load labels
df_labels_final = spark.read.json(output_path)
df_labels_final = df_labels_final.select("transaction_id", "fraud_label")

In [0]:
from pyspark.sql import functions as F
# Join and filter to 2010-2018
df = txn.join(df_labels_final, txn['id'] == df_labels_final['transaction_id'], how="inner") \
                 .filter(F.year("date") <= 2018)

In [0]:
display(df)

In [0]:
class_counts = df.groupBy("fraud_label") \
                 .count() \
                 .withColumn("percentage", 
                    F.round(F.col("count") / df.count() * 100, 4)) \
                 .orderBy("fraud_label")

class_counts.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_plot = class_counts.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Class Imbalance Profile (2010–2018)", fontsize=14, fontweight="bold")

# Plot 1: Count bar chart
axes[0].bar(df_plot["fraud_label"], df_plot["count"], 
            color=["steelblue", "crimson"])
axes[0].set_title("Transaction Count by Class")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{x/1e6:.1f}M"))
for i, row in df_plot.iterrows():
    axes[0].text(i, row["count"], f'{row["count"]:,}', 
                 ha="center", va="bottom", fontsize=10)

# Plot 2: Pie chart
axes[1].pie(df_plot["count"], labels=df_plot["fraud_label"], 
            autopct="%1.3f%%", colors=["steelblue", "crimson"],
            startangle=90)
axes[1].set_title("Class Proportion")

plt.tight_layout()
plt.show()

In [0]:
counts = class_counts.toPandas().set_index("fraud_label")["count"]
ratio = counts["No"] / counts["Yes"]
print(f"Imbalance ratio: {ratio:.0f}:1 (non-fraud:fraud)")

In [0]:
##### Temporal trends #####

In [0]:
yearly = df.groupBy(F.year("date").alias("year")) \
           .agg(
               F.count("*").alias("total"),
               F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
           ) \
           .withColumn("fraud_rate", F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
           .orderBy("year")

yearly.show()

In [0]:
df_yearly = yearly.toPandas()

fig, ax1 = plt.subplots(figsize=(12, 5))
fig.suptitle("Yearly Fraud Trends (2010–2018)", fontsize=14, fontweight="bold")

# Bar chart for transaction volume
ax1.bar(df_yearly["year"], df_yearly["total"], 
        color="steelblue", alpha=0.6, label="Total Transactions")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Transactions", color="steelblue")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{x/1e6:.1f}M"))

# Line chart for fraud rate on secondary axis
ax2 = ax1.twinx()
ax2.plot(df_yearly["year"], df_yearly["fraud_rate"], 
         color="crimson", marker="o", linewidth=2, label="Fraud Rate %")
ax2.set_ylabel("Fraud Rate (%)", color="crimson")

# Legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

In [0]:
monthly = df.groupBy(
               F.year("date").alias("year"),
               F.month("date").alias("month")
           ) \
           .agg(
               F.count("*").alias("total"),
               F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
           ) \
           .withColumn("fraud_rate", F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
           .orderBy("year", "month")

import pandas as pd

df_monthly = monthly.toPandas()
df_monthly["period"] = pd.to_datetime(
    df_monthly[["year", "month"]].assign(day=1))

In [0]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_monthly["period"], df_monthly["fraud_rate"], 
        color="crimson", linewidth=1.5)
ax.set_title("Monthly Fraud Rate (2010–2018)", fontsize=13)
ax.set_xlabel("Month")
ax.set_ylabel("Fraud Rate (%)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

In [0]:
dow = df.groupBy(F.dayofweek("date").alias("day_of_week")) \
        .agg(
            F.count("*").alias("total"),
            F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
        ) \
        .withColumn("fraud_rate", F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
        .orderBy("day_of_week")

hod = df.groupBy(F.hour("date").alias("hour")) \
        .agg(
            F.count("*").alias("total"),
            F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
        ) \
        .withColumn("fraud_rate", F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
        .orderBy("hour")

df_dow = dow.toPandas()
df_hod = hod.toPandas()

# Remap day numbers to labels (Spark: 1=Sunday, 7=Saturday)
day_labels = {1:"Sun", 2:"Mon", 3:"Tue", 4:"Wed", 5:"Thu", 6:"Fri", 7:"Sat"}
df_dow["day_label"] = df_dow["day_of_week"].map(day_labels)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Fraud Rate by Day of Week and Hour of Day", 
             fontsize=14, fontweight="bold")

# Day of week
axes[0].bar(df_dow["day_label"], df_dow["fraud_rate"], color="steelblue")
axes[0].set_title("By Day of Week")
axes[0].set_xlabel("Day")
axes[0].set_ylabel("Fraud Rate (%)")

# Hour of day
axes[1].bar(df_hod["hour"], df_hod["fraud_rate"], color="crimson")
axes[1].set_title("By Hour of Day")
axes[1].set_xlabel("Hour (0–23)")
axes[1].set_ylabel("Fraud Rate (%)")

plt.tight_layout()
plt.show()

In [0]:
fig, ax1 = plt.subplots(figsize=(12, 5))
fig.suptitle("Transaction Volume vs Fraud Count (2010 - 2018)", 
             fontsize=14, fontweight="bold")

ax1.bar(df_yearly["year"], df_yearly["total"], 
        color="steelblue", alpha=0.6, label="Total Transactions")
ax1.set_ylabel("Total Transactions", color="steelblue")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{x/1e6:.1f}M"))

ax2 = ax1.twinx()
ax2.plot(df_yearly["year"], df_yearly["fraud_count"], 
         color="crimson", marker="o", linewidth=2, label="Fraud Count")
ax2.set_ylabel("Fraud Count", color="crimson")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

In [0]:
### 2011 and 2017 exhibited very low fraud counts. This could be due to our decision to drop rows which had no validated labels resulting in a very small number of labeled transactions for those years. It may not be a genuine drop in fraud.

### we verify this below.

raw_yearly = txn.groupBy(F.year("date").alias("year"))\
            .count()\
            .withColumnRenamed("count", "raw_count")

labelled_yearly = df.groupBy(F.year("date").alias("year"))\
            .count()\
            .withColumnRenamed("count", "labelled_count")

raw_yearly = raw_yearly.join(labelled_yearly, on="year")\
            .withColumn("label_coverage",
                        F.round(F.col("labelled_count")/F.col("raw_count")*100, 2))\
            .orderBy("year")\
            .show()

the above verification disproves the theory. 

Label coverage is consistent at approximately 67% across all years, indicating that the anomalously low fraud counts in 2011 (37 cases) and 2017 (172 cases) are not attributable to missing labels. These years appear to reflect inconsistencies in the fraud labelling process within the source dataset. Both years are retained in the analysis, but their fraud rates are treated as unreliable and this limitation is acknowledged.

In [0]:
from pyspark.sql.types import DoubleType

df = df.withColumn("amount_clean", 
        F.regexp_replace(F.col("amount"), "[$,]", "").cast(DoubleType()))

In [0]:
amount_stats = df.groupBy("fraud_label") \
                 .agg(
                     F.count("*").alias("count"),
                     F.round(F.mean("amount_clean"), 2).alias("mean"),
                     F.round(F.stddev("amount_clean"), 2).alias("std"),
                     F.round(F.min("amount_clean"), 2).alias("min"),
                     F.expr("percentile(amount_clean, 0.25)").alias("p25"),
                     F.expr("percentile(amount_clean, 0.50)").alias("median"),
                     F.expr("percentile(amount_clean, 0.75)").alias("p75"),
                     F.round(F.max("amount_clean"), 2).alias("max")
                 ) \
                 .orderBy("fraud_label")

amount_stats.show()

In [0]:
negative_amounts = df.groupBy("fraud_label") \
                     .agg(
                         F.count("*").alias("total"),
                         F.sum((F.col("amount_clean") < 0).cast("int"))
                          .alias("negative_count")
                     ) \
                     .withColumn("negative_rate",
                         F.round(F.col("negative_count") / F.col("total") * 100, 4)) \
                     .orderBy("fraud_label")

negative_amounts.show()

In [0]:
# Filter to positive amounts only for bucketing
df_pos = df.filter(F.col("amount_clean") > 0)

# Define buckets
buckets = [0, 10, 25, 50, 100, 250, 500, 1000, 5000, 99999]
bucket_labels = ["$0-10", "$10-25", "$25-50", "$50-100", 
                 "$100-250", "$250-500", "$500-1K", "$1K-5K", "$5K+"]

# Assign buckets
df_bucketed = df_pos.withColumn("amount_bucket",
    F.when(F.col("amount_clean") < 10, "$0-10")
     .when(F.col("amount_clean") < 25, "$10-25")
     .when(F.col("amount_clean") < 50, "$25-50")
     .when(F.col("amount_clean") < 100, "$50-100")
     .when(F.col("amount_clean") < 250, "$100-250")
     .when(F.col("amount_clean") < 500, "$250-500")
     .when(F.col("amount_clean") < 1000, "$500-1K")
     .when(F.col("amount_clean") < 5000, "$1K-5K")
     .otherwise("$5K+")
)

bucket_dist = df_bucketed.groupBy("amount_bucket", "fraud_label") \
                         .count() \
                         .orderBy("amount_bucket")

# Pivot for easier comparison
bucket_pivot = bucket_dist.groupBy("amount_bucket") \
                          .pivot("fraud_label", ["No", "Yes"]) \
                          .sum("count") \
                          .withColumnRenamed("No", "non_fraud") \
                          .withColumnRenamed("Yes", "fraud")

# Add fraud rate per bucket
bucket_pivot = bucket_pivot.withColumn("fraud_rate",
    F.round(F.col("fraud") / (F.col("fraud") + F.col("non_fraud")) * 100, 4)) \
    .orderBy("amount_bucket")

bucket_pivot.show()

In [0]:
import numpy as np

df_buckets = bucket_pivot.toPandas()
df_stats = amount_stats.toPandas()

# Order buckets correctly
bucket_order = ["$0-10", "$10-25", "$25-50", "$50-100", 
                "$100-250", "$250-500", "$500-1K", "$1K-5K", "$5K+"]
df_buckets["amount_bucket"] = pd.Categorical(
    df_buckets["amount_bucket"], categories=bucket_order, ordered=True)
df_buckets = df_buckets.sort_values("amount_bucket")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Transaction Amount Analysis (2010–2018)", 
             fontsize=14, fontweight="bold")

# Plot 1: Mean amount by class
classes = df_stats["fraud_label"]
means = df_stats["mean"]
axes[0, 0].bar(classes, means, color=["steelblue", "crimson"])
axes[0, 0].set_title("Mean Transaction Amount by Class")
axes[0, 0].set_xlabel("Class")
axes[0, 0].set_ylabel("Mean Amount ($)")
for i, v in enumerate(means):
    axes[0, 0].text(i, v, f'${v:,.2f}', ha="center", va="bottom")

# Plot 2: Transaction count by amount bucket
x = np.arange(len(df_buckets))
width = 0.35
axes[0, 1].bar(x - width/2, df_buckets["non_fraud"], width, 
               label="Non-Fraud", color="steelblue", alpha=0.7)
axes[0, 1].bar(x + width/2, df_buckets["fraud"], width, 
               label="Fraud", color="crimson", alpha=0.7)
axes[0, 1].set_title("Transaction Count by Amount Bucket")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(df_buckets["amount_bucket"], rotation=45, ha="right")
axes[0, 1].set_ylabel("Count")
axes[0, 1].legend()

# Plot 3: Fraud rate by amount bucket
axes[1, 0].bar(df_buckets["amount_bucket"], df_buckets["fraud_rate"], 
               color="crimson", alpha=0.7)
axes[1, 0].set_title("Fraud Rate by Amount Bucket")
axes[1, 0].set_xlabel("Amount Bucket")
axes[1, 0].set_ylabel("Fraud Rate (%)")
axes[1, 0].tick_params(axis="x", rotation=45)

# Plot 4: Box plot comparison (use sample for this)
# Replace the applyInPandas block with this
df_sample = df.filter(F.col("amount_clean") > 0) \
              .sample(fraction=0.005, seed=42) \
              .select("fraud_label", "amount_clean") \
              .toPandas()

fraud_amounts = df_sample[df_sample["fraud_label"] == "Yes"]["amount_clean"]
non_fraud_amounts = df_sample[df_sample["fraud_label"] == "No"]["amount_clean"]

axes[1, 1].boxplot([non_fraud_amounts, fraud_amounts], 
                   labels=["Non-Fraud", "Fraud"],
                   patch_artist=True,
                   boxprops=dict(facecolor="steelblue", alpha=0.7),
                   medianprops=dict(color="crimson", linewidth=2))
axes[1, 1].set_title("Amount Distribution by Class (Sampled)")
axes[1, 1].set_ylabel("Amount ($)")

plt.tight_layout()
plt.show()

Key insight: The separation between classes is real but not clean — fraud has a higher mean and median but a very large standard deviation. This means amount alone is not sufficient to flag fraud — there are fraudulent transactions at every level. However it remains a useful feature, particularly in combination with others.
The high std in fraud ($219 vs $81) suggests fraudsters operate across a wide range of amounts — from small card-testing transactions to large single hits.

In [0]:
payment = df.groupBy("use_chip") \
            .agg(
                F.count("*").alias("total"),
                F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
            ) \
            .withColumn("fraud_rate", 
                F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
            .withColumn("pct_of_transactions",
                F.round(F.col("total") / df.count() * 100, 2)) \
            .orderBy(F.desc("fraud_rate"))

payment.show()

In [0]:
payment_yearly = df.groupBy(
                    F.year("date").alias("year"), 
                    "use_chip"
                 ) \
                 .agg(
                     F.count("*").alias("total"),
                     F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
                 ) \
                 .withColumn("fraud_rate",
                     F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                 .orderBy("year", "use_chip")

payment_yearly.show(30)

In [0]:
df_payment = payment.toPandas()
df_payment_yearly = payment_yearly.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Payment Method Analysis (2010–2018)", 
             fontsize=14, fontweight="bold")

# Plot 1: Transaction share by payment method
axes[0].bar(df_payment["use_chip"], df_payment["pct_of_transactions"],
            color=["steelblue", "crimson", "seagreen"])
axes[0].set_title("Transaction Share by Payment Method")
axes[0].set_xlabel("Payment Method")
axes[0].set_ylabel("% of Total Transactions")
axes[0].tick_params(axis="x", rotation=15)
for i, row in df_payment.iterrows():
    axes[0].text(i, row["pct_of_transactions"], 
                 f'{row["pct_of_transactions"]}%',
                 ha="center", va="bottom", fontsize=9)

# Plot 2: Fraud rate by payment method
axes[1].bar(df_payment["use_chip"], df_payment["fraud_rate"],
            color=["steelblue", "crimson", "seagreen"])
axes[1].set_title("Fraud Rate by Payment Method")
axes[1].set_xlabel("Payment Method")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].axhline(y=0.1471, color="black", linestyle="--", 
                linewidth=1, label="Dataset Average (0.1471%)")
axes[1].legend()
for i, row in df_payment.iterrows():
    axes[1].text(i, row["fraud_rate"],
                 f'{row["fraud_rate"]}%',
                 ha="center", va="bottom", fontsize=9)

# Plot 3: Fraud rate by payment method over time
for method in df_payment_yearly["use_chip"].unique():
    subset = df_payment_yearly[df_payment_yearly["use_chip"] == method]
    axes[2].plot(subset["year"], subset["fraud_rate"], 
                 marker="o", linewidth=2, label=method)
axes[2].set_title("Fraud Rate by Payment Method Over Time")
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].legend(fontsize=8)
axes[2].axhline(y=0.1471, color="black", linestyle="--", 
                linewidth=1, label="Dataset Average")

plt.tight_layout()
plt.show()

In [0]:
# # load mcc codes
mcc = spark.read.format("json") \
    .option("multiline", "true") \
    .option("inferSchema", "true") \
    .load(s3_bucket_path + "mcc_codes.json")
mcc_cols = mcc.columns

In [0]:
# Create array of structs with code and description
from pyspark.sql.functions import array, struct, lit, explode

# Build an array of (code, description) pairs
kvs = [struct(lit(c).alias("code"), col(c).alias("description")) for c in mcc_cols]

# Explode to convert columns to rows
mcc_df = mcc.select(explode(array(*kvs)).alias("data")) \
    .select("data.code", "data.description")

# mcc_df.show()

In [0]:
mcc_data = mcc_df.withColumnRenamed("code", "mcc") \
                 .withColumnRenamed("description", "mcc_description")

df_mcc = df.join(mcc_data, on="mcc", how="left")

In [0]:
# Filter out rare MCCs with fewer than 1000 transactions
mcc_fraud = df_mcc.groupBy("mcc", "mcc_description") \
                  .agg(
                      F.count("*").alias("total"),
                      F.sum((F.col("fraud_label") == "Yes").cast("int")).alias("fraud_count")
                  ) \
                  .withColumn("fraud_rate",
                      F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                  .filter(F.col("total") >= 1000) \
                  .orderBy(F.desc("fraud_rate"))

# Top 15 highest fraud rate MCCs
mcc_fraud.show(15)

# Top 15 highest fraud volume MCCs
mcc_fraud.orderBy(F.desc("fraud_count")).show(15)

In [0]:
df_mcc_rate = mcc_fraud.orderBy(F.desc("fraud_rate")).limit(15).toPandas()
df_mcc_vol = mcc_fraud.orderBy(F.desc("fraud_count")).limit(15).toPandas()

fig, axes = plt.subplots(2, 1, figsize=(16, 14))
fig.suptitle("MCC Fraud Analysis (2010–2018)", 
             fontsize=14, fontweight="bold")

# Plot 1: Top 15 MCCs by fraud rate
axes[0].barh(df_mcc_rate["mcc_description"], 
             df_mcc_rate["fraud_rate"],
             color="crimson", alpha=0.7)
axes[0].set_title("Top 15 MCCs by Fraud Rate (min 1,000 transactions)")
axes[0].set_xlabel("Fraud Rate (%)")
axes[0].axvline(x=0.1471, color="black", linestyle="--",
                linewidth=1, label="Dataset Average (0.1471%)")
axes[0].legend()
axes[0].invert_yaxis()

# Plot 2: Top 15 MCCs by fraud volume
axes[1].barh(df_mcc_vol["mcc_description"],
             df_mcc_vol["fraud_count"],
             color="steelblue", alpha=0.7)
axes[1].set_title("Top 15 MCCs by Fraud Volume")
axes[1].set_xlabel("Fraud Count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
user_data = spark.read.csv(s3_bucket_path + "users_data.csv", header=True, inferSchema=True)

df_geo = df.join(user_data.select(
                    "id", "latitude", "longitude"
                 ).withColumnRenamed("id", "client_id"),
                 on="client_id", how="left")

In [0]:
state_fraud = df_geo.filter(F.col("merchant_state").isNotNull()) \
                    .groupBy("merchant_state") \
                    .agg(
                        F.count("*").alias("total"),
                        F.sum((F.col("fraud_label") == "Yes").cast("int"))
                         .alias("fraud_count")
                    ) \
                    .withColumn("fraud_rate",
                        F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                    .filter(F.col("total") >= 1000) \
                    .orderBy(F.desc("fraud_rate"))

# Top 15 states by fraud rate
state_fraud.show(15)

# Top 15 states by fraud volume
state_fraud.orderBy(F.desc("fraud_count")).show(15)

In [0]:
# Look at Italy transactions in detail
italy = df_geo.filter(F.col("merchant_state") == "Italy")

# Check payment method distribution
italy.groupBy("use_chip").count().show()

# Check amount distribution
italy.select(F.mean("amount_clean"), 
             F.stddev("amount_clean"),
             F.min("amount_clean"),
             F.max("amount_clean")).show()

# Check year distribution
italy.groupBy(F.year("date").alias("year")).count().orderBy("year").show()

# Check merchant city
italy.groupBy("merchant_city").count().orderBy(F.desc("count")).show(10)

# Check MCC
italy.groupBy("mcc").count().orderBy(F.desc("count")).show(10)

# Look up MCC descriptions for Italy transactions
italy_mccs = italy.groupBy("mcc").count().orderBy(F.desc("count"))
italy_mccs.join(mcc_data, on="mcc", how="left").show(10)

Transactions with merchant_state = 'Italy' were identified as a data quality anomaly — 3,108 physical chip and swipe transactions all mapping to a single city (Rome) with everyday MCC categories and a 54.7% fraud rate. This pattern is inconsistent with genuine fraud behaviour and is attributed to systematic mislabelling in the source dataset. These 3,108 transactions (including 1,701 fraud labels representing 14.2% of total fraud cases) were excluded from all modelling and analysis

In [0]:
# Load state from user address for comparison
# We'll use a simplified approach — flag online transactions 
# and transactions where merchant_state is null (likely foreign/online)

df_geo = df_geo.withColumn("is_cross_border",
    F.when(
        (F.col("merchant_state").isNull()) | 
        (F.col("merchant_state") == ""), 1
    ).otherwise(0))

cross_border = df_geo.groupBy("is_cross_border") \
                     .agg(
                         F.count("*").alias("total"),
                         F.sum((F.col("fraud_label") == "Yes").cast("int"))
                          .alias("fraud_count")
                     ) \
                     .withColumn("fraud_rate",
                         F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                     .orderBy("is_cross_border")

cross_border.show()

In [0]:
from pyspark.sql.functions import radians, cos, sin, asin, sqrt

# Haversine distance formula
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 3959  # Earth radius in miles
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

# Load zip code lat/lon reference 
# (derive from merchant zip in transaction data)
# For now use a simplified flag: null merchant state = likely remote/online
df_geo = df_geo.withColumn("customer_lat", F.col("latitude").cast("double")) \
               .withColumn("customer_lon", F.col("longitude").cast("double"))

# Flag transactions where merchant zip is null (no physical location)
df_geo = df_geo.withColumn("no_physical_location",
    F.when(F.col("zip").isNull(), 1).otherwise(0))

location_fraud = df_geo.groupBy("no_physical_location") \
                       .agg(
                           F.count("*").alias("total"),
                           F.sum((F.col("fraud_label") == "Yes").cast("int"))
                            .alias("fraud_count")
                       ) \
                       .withColumn("fraud_rate",
                           F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                       .orderBy("no_physical_location")

location_fraud.show()

In [0]:
# Get top 20 states for visualisation
df_state_rate = state_fraud.limit(20).toPandas()
df_state_vol = state_fraud.orderBy(F.desc("fraud_count")).limit(20).toPandas()
df_cross = cross_border.toPandas()
df_location = location_fraud.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Geographic Fraud Analysis (2010–2018)", 
             fontsize=14, fontweight="bold")

# Plot 1: Top 20 states by fraud rate
axes[0, 0].barh(df_state_rate["merchant_state"], 
                df_state_rate["fraud_rate"],
                color="crimson", alpha=0.7)
axes[0, 0].set_title("Top 20 States by Fraud Rate (min 1,000 txns)")
axes[0, 0].set_xlabel("Fraud Rate (%)")
axes[0, 0].axvline(x=0.1471, color="black", linestyle="--",
                   linewidth=1, label="Dataset Average")
axes[0, 0].legend()
axes[0, 0].invert_yaxis()

# Plot 2: Top 20 states by fraud volume
axes[0, 1].barh(df_state_vol["merchant_state"],
                df_state_vol["fraud_count"],
                color="steelblue", alpha=0.7)
axes[0, 1].set_title("Top 20 States by Fraud Volume")
axes[0, 1].set_xlabel("Fraud Count")
axes[0, 1].invert_yaxis()

# Plot 3: Cross-border fraud rate
labels = ["In-State", "Cross-Border/Online"]
axes[1, 0].bar(labels,
               df_cross["fraud_rate"],
               color=["steelblue", "crimson"], alpha=0.7)
axes[1, 0].set_title("Fraud Rate: In-State vs Cross-Border")
axes[1, 0].set_ylabel("Fraud Rate (%)")
axes[1, 0].axhline(y=0.1471, color="black", linestyle="--",
                   linewidth=1, label="Dataset Average")
axes[1, 0].legend()
for i, row in df_cross.iterrows():
    axes[1, 0].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom")

# Plot 4: Physical vs no physical location
labels2 = ["Has Physical Location", "No Physical Location"]
axes[1, 1].bar(labels2,
               df_location["fraud_rate"],
               color=["steelblue", "crimson"], alpha=0.7)
axes[1, 1].set_title("Fraud Rate: Physical vs No Physical Location")
axes[1, 1].set_ylabel("Fraud Rate (%)")
axes[1, 1].axhline(y=0.1471, color="black", linestyle="--",
                   linewidth=1, label="Dataset Average")
axes[1, 1].legend()
for i, row in df_location.iterrows():
    axes[1, 1].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [0]:
# Correctly exclude Italy while preserving NULL merchant states
df_clean = df_geo.filter(
    F.col("merchant_state").isNull() | 
    (F.col("merchant_state") != "Italy")
)

# Verify
print("After correct exclusion:")
print(f"Total: {df_clean.count():,}")
print(f"Fraud: {df_clean.filter(F.col('fraud_label') == 'Yes').count():,}")

In [0]:
state_fraud = df_clean.filter(F.col("merchant_state").isNotNull()) \
                      .groupBy("merchant_state") \
                      .agg(
                          F.count("*").alias("total"),
                          F.sum((F.col("fraud_label") == "Yes").cast("int"))
                           .alias("fraud_count")
                      ) \
                      .withColumn("fraud_rate",
                          F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                      .filter(F.col("total") >= 1000) \
                      .orderBy(F.desc("fraud_rate"))

# Top 15 by fraud rate
state_fraud.show(15)

# Top 15 by fraud volume
state_fraud.orderBy(F.desc("fraud_count")).show(15)

In [0]:
cross_border = df_clean.groupBy("is_cross_border") \
                       .agg(
                           F.count("*").alias("total"),
                           F.sum((F.col("fraud_label") == "Yes").cast("int"))
                            .alias("fraud_count")
                       ) \
                       .withColumn("fraud_rate",
                           F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                       .orderBy("is_cross_border")

cross_border.show()

In [0]:
location_fraud = df_clean.groupBy("no_physical_location") \
                         .agg(
                             F.count("*").alias("total"),
                             F.sum((F.col("fraud_label") == "Yes").cast("int"))
                              .alias("fraud_count")
                         ) \
                         .withColumn("fraud_rate",
                             F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                         .orderBy("no_physical_location")

location_fraud.show()

In [0]:
total = df_clean.count()
fraud_total = df_clean.filter(F.col("fraud_label") == "Yes").count()

# Use Python built-in round explicitly
import builtins
new_fraud_rate = builtins.round(fraud_total / total * 100, 4)
new_imbalance_ratio = builtins.round((total - fraud_total) / fraud_total)

print(f"New total transactions: {total:,}")
print(f"New fraud count: {fraud_total:,}")
print(f"New fraud rate: {new_fraud_rate}%")
print(f"New imbalance ratio: {new_imbalance_ratio}:1")

In [0]:
df_state_rate = state_fraud.limit(20).toPandas()
df_state_vol = state_fraud.orderBy(F.desc("fraud_count")).limit(20).toPandas()
df_cross = cross_border.toPandas()
df_location = location_fraud.toPandas()

# Updated fraud rate after Italy exclusion
DATASET_AVG = new_fraud_rate

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Geographic Fraud Analysis (2010–2018, Italy Excluded)", 
             fontsize=14, fontweight="bold")

# Plot 1: Top 20 states by fraud rate
axes[0, 0].barh(df_state_rate["merchant_state"],
                df_state_rate["fraud_rate"],
                color="crimson", alpha=0.7)
axes[0, 0].set_title("Top 20 States by Fraud Rate (min 1,000 txns)")
axes[0, 0].set_xlabel("Fraud Rate (%)")
axes[0, 0].axvline(x=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0, 0].legend()
axes[0, 0].invert_yaxis()

# Plot 2: Top 20 states by fraud volume
axes[0, 1].barh(df_state_vol["merchant_state"],
                df_state_vol["fraud_count"],
                color="steelblue", alpha=0.7)
axes[0, 1].set_title("Top 20 States by Fraud Volume")
axes[0, 1].set_xlabel("Fraud Count")
axes[0, 1].invert_yaxis()

# Plot 3: Cross-border fraud rate
labels = ["In-State/Known", "Cross-Border/Online"]
axes[1, 0].bar(labels,
               df_cross["fraud_rate"],
               color=["steelblue", "crimson"], alpha=0.7)
axes[1, 0].set_title("Fraud Rate: In-State vs Cross-Border")
axes[1, 0].set_ylabel("Fraud Rate (%)")
axes[1, 0].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 0].legend()
for i, row in df_cross.iterrows():
    axes[1, 0].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom")

# Plot 4: Physical vs no physical location
labels2 = ["Has Physical Location", "No Physical Location"]
axes[1, 1].bar(labels2,
               df_location["fraud_rate"],
               color=["steelblue", "crimson"], alpha=0.7)
axes[1, 1].set_title("Fraud Rate: Physical vs No Physical Location")
axes[1, 1].set_ylabel("Fraud Rate (%)")
axes[1, 1].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 1].legend()
for i, row in df_location.iterrows():
    axes[1, 1].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [0]:
# How many customers experienced fraud, and how many times?
customer_fraud = df_clean.groupBy("client_id") \
                         .agg(
                             F.count("*").alias("total_txns"),
                             F.sum((F.col("fraud_label") == "Yes").cast("int"))
                              .alias("fraud_count")
                         ) \
                         .withColumn("is_fraud_victim",
                             F.when(F.col("fraud_count") > 0, 1).otherwise(0))

# Overall victim stats
customer_fraud.groupBy("is_fraud_victim") \
              .agg(
                  F.count("*").alias("customer_count"),
                  F.sum("total_txns").alias("total_txns"),
                  F.sum("fraud_count").alias("total_fraud")
              ).show()

# Distribution of fraud incidents per victim
customer_fraud.filter(F.col("is_fraud_victim") == 1) \
              .groupBy("fraud_count") \
              .count() \
              .orderBy("fraud_count") \
              .show(20)

In [0]:
# How many customers have multiple fraud incidents?
repeat_victims = customer_fraud.filter(F.col("fraud_count") > 1) \
                               .agg(
                                   F.count("*").alias("repeat_victim_count"),
                                   F.max("fraud_count").alias("max_fraud_incidents"),
                                   F.mean("fraud_count").alias("avg_fraud_incidents")
                               )
repeat_victims.show()

# Top 10 most targeted customers
customer_fraud.filter(F.col("is_fraud_victim") == 1) \
              .orderBy(F.desc("fraud_count")) \
              .show(10)

In [0]:
# Join full user data
user_data =  spark.read.csv(s3_bucket_path + "users_data.csv", header=True, inferSchema=True)


df_customer = df_clean.join(
                  user_data.withColumnRenamed("id", "client_id"),
                  on="client_id", how="left"
              )

In [0]:
# Bin customers into age groups
df_customer = df_customer.withColumn("age_group",
    F.when(F.col("current_age") < 25, "18-24")
     .when(F.col("current_age") < 35, "25-34")
     .when(F.col("current_age") < 45, "35-44")
     .when(F.col("current_age") < 55, "45-54")
     .when(F.col("current_age") < 65, "55-64")
     .otherwise("65+"))

age_fraud = df_customer.groupBy("age_group") \
                       .agg(
                           F.count("*").alias("total"),
                           F.sum((F.col("fraud_label") == "Yes").cast("int"))
                            .alias("fraud_count")
                       ) \
                       .withColumn("fraud_rate",
                           F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                       .orderBy("age_group")

age_fraud.show()

In [0]:
# Clean yearly_income column (remove $ and commas)
df_customer = df_customer.withColumn("income_clean",
    F.regexp_replace(F.col("yearly_income"), "[$,]", "").cast("double"))

# Bin into income groups
df_customer = df_customer.withColumn("income_group",
    F.when(F.col("income_clean") < 30000, "Under $30K")
     .when(F.col("income_clean") < 60000, "$30K-$60K")
     .when(F.col("income_clean") < 100000, "$60K-$100K")
     .when(F.col("income_clean") < 150000, "$100K-$150K")
     .otherwise("$150K+"))

income_fraud = df_customer.groupBy("income_group") \
                          .agg(
                              F.count("*").alias("total"),
                              F.sum((F.col("fraud_label") == "Yes").cast("int"))
                               .alias("fraud_count")
                          ) \
                          .withColumn("fraud_rate",
                              F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                          .orderBy("income_group")

income_fraud.show()

In [0]:
df_customer = df_customer.withColumn("credit_band",
    F.when(F.col("credit_score") < 580, "Poor (<580)")
     .when(F.col("credit_score") < 670, "Fair (580-669)")
     .when(F.col("credit_score") < 740, "Good (670-739)")
     .when(F.col("credit_score") < 800, "Very Good (740-799)")
     .otherwise("Exceptional (800+)"))

credit_fraud = df_customer.groupBy("credit_band") \
                          .agg(
                              F.count("*").alias("total"),
                              F.sum((F.col("fraud_label") == "Yes").cast("int"))
                               .alias("fraud_count")
                          ) \
                          .withColumn("fraud_rate",
                              F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                          .orderBy("credit_band")

credit_fraud.show()

In [0]:
gender_fraud = df_customer.groupBy("gender") \
                          .agg(
                              F.count("*").alias("total"),
                              F.sum((F.col("fraud_label") == "Yes").cast("int"))
                               .alias("fraud_count")
                          ) \
                          .withColumn("fraud_rate",
                              F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                          .orderBy("gender")

gender_fraud.show()

In [0]:
df_age = age_fraud.toPandas()
df_income = income_fraud.toPandas()
df_credit = credit_fraud.toPandas()
df_gender = gender_fraud.toPandas()

# Order categories correctly
age_order = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
income_order = ["Under $30K", "$30K-$60K", "$60K-$100K", 
                "$100K-$150K", "$150K+"]
credit_order = ["Poor (<580)", "Fair (580-669)", "Good (670-739)",
                "Very Good (740-799)", "Exceptional (800+)"]

df_age["age_group"] = pd.Categorical(
    df_age["age_group"], categories=age_order, ordered=True)
df_age = df_age.sort_values("age_group")

df_income["income_group"] = pd.Categorical(
    df_income["income_group"], categories=income_order, ordered=True)
df_income = df_income.sort_values("income_group")

df_credit["credit_band"] = pd.Categorical(
    df_credit["credit_band"], categories=credit_order, ordered=True)
df_credit = df_credit.sort_values("credit_band")

DATASET_AVG = 0.1263

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Customer-Level Fraud Analysis (2010–2018)", 
             fontsize=14, fontweight="bold")

# Plot 1: Fraud rate by age group
axes[0, 0].bar(df_age["age_group"], df_age["fraud_rate"],
               color="crimson", alpha=0.7)
axes[0, 0].set_title("Fraud Rate by Age Group")
axes[0, 0].set_xlabel("Age Group")
axes[0, 0].set_ylabel("Fraud Rate (%)")
axes[0, 0].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0, 0].legend()
for i, row in df_age.iterrows():
    axes[0, 0].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 2: Fraud rate by income group
axes[0, 1].bar(df_income["income_group"], df_income["fraud_rate"],
               color="steelblue", alpha=0.7)
axes[0, 1].set_title("Fraud Rate by Income Group")
axes[0, 1].set_xlabel("Income Group")
axes[0, 1].set_ylabel("Fraud Rate (%)")
axes[0, 1].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0, 1].legend()
axes[0, 1].tick_params(axis="x", rotation=15)
for i, row in df_income.iterrows():
    axes[0, 1].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 3: Fraud rate by credit score band
axes[1, 0].bar(df_credit["credit_band"], df_credit["fraud_rate"],
               color="seagreen", alpha=0.7)
axes[1, 0].set_title("Fraud Rate by Credit Score Band")
axes[1, 0].set_xlabel("Credit Score Band")
axes[1, 0].set_ylabel("Fraud Rate (%)")
axes[1, 0].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 0].legend()
axes[1, 0].tick_params(axis="x", rotation=15)
for i, row in df_credit.iterrows():
    axes[1, 0].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 4: Fraud rate by gender
axes[1, 1].bar(df_gender["gender"], df_gender["fraud_rate"],
               color="mediumpurple", alpha=0.7)
axes[1, 1].set_title("Fraud Rate by Gender")
axes[1, 1].set_xlabel("Gender")
axes[1, 1].set_ylabel("Fraud Rate (%)")
axes[1, 1].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 1].legend()
for i, row in df_gender.iterrows():
    axes[1, 1].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

In [0]:
card_data = spark.read.csv(s3_bucket_path + "cards_data.csv", header=True, inferSchema=True)

df_card = df_clean.join(
              card_data.withColumnRenamed("id", "card_id"),
              on="card_id", how="left"
          )

In [0]:
brand_fraud = df_card.groupBy("card_brand") \
                     .agg(
                         F.count("*").alias("total"),
                         F.sum((F.col("fraud_label") == "Yes").cast("int"))
                          .alias("fraud_count")
                     ) \
                     .withColumn("fraud_rate",
                         F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                     .orderBy(F.desc("fraud_rate"))

brand_fraud.show()

In [0]:
type_fraud = df_card.groupBy("card_type") \
                    .agg(
                        F.count("*").alias("total"),
                        F.sum((F.col("fraud_label") == "Yes").cast("int"))
                         .alias("fraud_count")
                    ) \
                    .withColumn("fraud_rate",
                        F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                    .orderBy(F.desc("fraud_rate"))

type_fraud.show()

In [0]:
chip_fraud = df_card.groupBy("has_chip") \
                    .agg(
                        F.count("*").alias("total"),
                        F.sum((F.col("fraud_label") == "Yes").cast("int"))
                         .alias("fraud_count")
                    ) \
                    .withColumn("fraud_rate",
                        F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                    .orderBy(F.desc("fraud_rate"))

chip_fraud.show()

In [0]:
darkweb_fraud = df_card.groupBy("card_on_dark_web") \
                       .agg(
                           F.count("*").alias("total"),
                           F.sum((F.col("fraud_label") == "Yes").cast("int"))
                            .alias("fraud_count")
                       ) \
                       .withColumn("fraud_rate",
                           F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                       .orderBy(F.desc("fraud_rate"))

darkweb_fraud.show()

In [0]:
# Clean credit limit column
df_card = df_card.withColumn("credit_limit_clean",
    F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double"))

# Bin into credit limit groups
df_card = df_card.withColumn("credit_limit_group",
    F.when(F.col("credit_limit_clean") < 5000, "Under $5K")
     .when(F.col("credit_limit_clean") < 10000, "$5K-$10K")
     .when(F.col("credit_limit_clean") < 20000, "$10K-$20K")
     .when(F.col("credit_limit_clean") < 30000, "$20K-$30K")
     .otherwise("$30K+"))

limit_fraud = df_card.groupBy("credit_limit_group") \
                     .agg(
                         F.count("*").alias("total"),
                         F.sum((F.col("fraud_label") == "Yes").cast("int"))
                          .alias("fraud_count")
                     ) \
                     .withColumn("fraud_rate",
                         F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                     .orderBy("credit_limit_group")

limit_fraud.show()

In [0]:
cards_issued_fraud = df_card.groupBy("num_cards_issued") \
                            .agg(
                                F.count("*").alias("total"),
                                F.sum((F.col("fraud_label") == "Yes").cast("int"))
                                 .alias("fraud_count")
                            ) \
                            .withColumn("fraud_rate",
                                F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                            .orderBy("num_cards_issued")

cards_issued_fraud.show()

In [0]:
df_brand = brand_fraud.toPandas()
df_type = type_fraud.toPandas()
df_chip = chip_fraud.toPandas()
df_darkweb = darkweb_fraud.toPandas()
df_limit = limit_fraud.toPandas()

limit_order = ["Under $5K", "$5K-$10K", "$10K-$20K", "$20K-$30K", "$30K+"]
df_limit["credit_limit_group"] = pd.Categorical(
    df_limit["credit_limit_group"], categories=limit_order, ordered=True)
df_limit = df_limit.sort_values("credit_limit_group")

DATASET_AVG = 0.1263

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Card-Level Fraud Analysis (2010–2018)",
             fontsize=14, fontweight="bold")

# Plot 1: Card brand
axes[0, 0].bar(df_brand["card_brand"], df_brand["fraud_rate"],
               color="crimson", alpha=0.7)
axes[0, 0].set_title("Fraud Rate by Card Brand")
axes[0, 0].set_xlabel("Card Brand")
axes[0, 0].set_ylabel("Fraud Rate (%)")
axes[0, 0].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0, 0].legend(fontsize=8)
for i, row in df_brand.iterrows():
    axes[0, 0].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 2: Card type
axes[0, 1].bar(df_type["card_type"], df_type["fraud_rate"],
               color="steelblue", alpha=0.7)
axes[0, 1].set_title("Fraud Rate by Card Type")
axes[0, 1].set_xlabel("Card Type")
axes[0, 1].set_ylabel("Fraud Rate (%)")
axes[0, 1].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0, 1].legend(fontsize=8)
axes[0, 1].tick_params(axis="x", rotation=15)
for i, row in df_type.iterrows():
    axes[0, 1].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 3: Chip capability
axes[0, 2].bar(df_chip["has_chip"], df_chip["fraud_rate"],
               color="seagreen", alpha=0.7)
axes[0, 2].set_title("Fraud Rate by Chip Capability")
axes[0, 2].set_xlabel("Has Chip")
axes[0, 2].set_ylabel("Fraud Rate (%)")
axes[0, 2].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0, 2].legend(fontsize=8)
for i, row in df_chip.iterrows():
    axes[0, 2].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 4: Dark web exposure
axes[1, 0].bar(df_darkweb["card_on_dark_web"], df_darkweb["fraud_rate"],
               color="mediumpurple", alpha=0.7)
axes[1, 0].set_title("Fraud Rate by Dark Web Exposure")
axes[1, 0].set_xlabel("Card on Dark Web")
axes[1, 0].set_ylabel("Fraud Rate (%)")
axes[1, 0].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 0].legend(fontsize=8)
for i, row in df_darkweb.iterrows():
    axes[1, 0].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 5: Credit limit group
axes[1, 1].bar(df_limit["credit_limit_group"], df_limit["fraud_rate"],
               color="darkorange", alpha=0.7)
axes[1, 1].set_title("Fraud Rate by Credit Limit")
axes[1, 1].set_xlabel("Credit Limit Group")
axes[1, 1].set_ylabel("Fraud Rate (%)")
axes[1, 1].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 1].legend(fontsize=8)
axes[1, 1].tick_params(axis="x", rotation=15)
for i, row in df_limit.iterrows():
    axes[1, 1].text(i, row["fraud_rate"],
                    f'{row["fraud_rate"]}%',
                    ha="center", va="bottom", fontsize=8)

# Plot 6: Cards issued
df_cards_issued = cards_issued_fraud.toPandas()
axes[1, 2].bar(df_cards_issued["num_cards_issued"].astype(str), 
               df_cards_issued["fraud_rate"],
               color="teal", alpha=0.7)
axes[1, 2].set_title("Fraud Rate by Number of Cards Issued")
axes[1, 2].set_xlabel("Number of Cards Issued")
axes[1, 2].set_ylabel("Fraud Rate (%)")
axes[1, 2].axhline(y=DATASET_AVG, color="black", linestyle="--",
                   linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1, 2].legend(fontsize=8)
axes[1, 2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# First examine what values exist in errors field
df_clean.groupBy("errors") \
        .count() \
        .orderBy(F.desc("count")) \
        .show(20)

In [0]:
# Create binary error flag
df_clean = df_clean.withColumn("has_error",
    F.when(F.col("errors").isNull() | 
           (F.col("errors") == ""), 0).otherwise(1))

# Fraud rate: error vs no error
error_fraud = df_clean.groupBy("has_error") \
                      .agg(
                          F.count("*").alias("total"),
                          F.sum((F.col("fraud_label") == "Yes").cast("int"))
                           .alias("fraud_count")
                      ) \
                      .withColumn("fraud_rate",
                          F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                      .orderBy("has_error")

error_fraud.show()

In [0]:
# Fraud rate per specific error type
error_type_fraud = df_clean.filter(F.col("has_error") == 1) \
                           .groupBy("errors") \
                           .agg(
                               F.count("*").alias("total"),
                               F.sum((F.col("fraud_label") == "Yes").cast("int"))
                                .alias("fraud_count")
                           ) \
                           .withColumn("fraud_rate",
                               F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
                           .orderBy(F.desc("fraud_rate"))

error_type_fraud.show(20)

In [0]:
df_error = error_fraud.toPandas()
df_error_type = error_type_fraud.toPandas()

DATASET_AVG = 0.1263

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Error Field Fraud Analysis (2010–2018)",
             fontsize=14, fontweight="bold")

# Plot 1: Error vs no error fraud rate
labels = ["No Error", "Has Error"]
axes[0].bar(labels, df_error["fraud_rate"],
            color=["steelblue", "crimson"], alpha=0.7)
axes[0].set_title("Fraud Rate: Error vs No Error")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].axhline(y=DATASET_AVG, color="black", linestyle="--",
                linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[0].legend()
for i, row in df_error.iterrows():
    axes[0].text(i, row["fraud_rate"],
                 f'{row["fraud_rate"]}%',
                 ha="center", va="bottom", fontsize=10)

# Plot 2: Fraud rate by error type
axes[1].barh(df_error_type["errors"],
             df_error_type["fraud_rate"],
             color="crimson", alpha=0.7)
axes[1].set_title("Fraud Rate by Error Type")
axes[1].set_xlabel("Fraud Rate (%)")
axes[1].axvline(x=DATASET_AVG, color="black", linestyle="--",
                linewidth=1, label=f"Dataset Average ({DATASET_AVG}%)")
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import unix_timestamp

# Convert date to timestamp in seconds
df_clean = df_clean.withColumn("ts", 
    unix_timestamp(F.col("date").cast("timestamp")))

# Define window specs per customer ordered by timestamp
# Look-back windows (in seconds)
one_hour = 3600
one_day = 86400
seven_days = 604800

# Window per customer, ordered by time
w_1h = Window.partitionBy("client_id") \
             .orderBy("ts") \
             .rangeBetween(-one_hour, -1)

w_1d = Window.partitionBy("client_id") \
             .orderBy("ts") \
             .rangeBetween(-one_day, -1)

w_7d = Window.partitionBy("client_id") \
             .orderBy("ts") \
             .rangeBetween(-seven_days, -1)

In [0]:
# Transaction count in each window
df_clean = df_clean.withColumn("txn_count_1h",
                    F.count("*").over(w_1h)) \
                   .withColumn("txn_count_1d",
                    F.count("*").over(w_1d)) \
                   .withColumn("txn_count_7d",
                    F.count("*").over(w_7d))

# Amount sum in each window
df_clean = df_clean.withColumn("amount_sum_1h",
                    F.sum("amount_clean").over(w_1h)) \
                   .withColumn("amount_sum_1d",
                    F.sum("amount_clean").over(w_1d)) \
                   .withColumn("amount_sum_7d",
                    F.sum("amount_clean").over(w_7d))

# Fill nulls for first transactions (no prior history)
velocity_cols = ["txn_count_1h", "txn_count_1d", "txn_count_7d",
                 "amount_sum_1h", "amount_sum_1d", "amount_sum_7d"]

for col in velocity_cols:
    df_clean = df_clean.withColumn(col, 
                   F.coalesce(F.col(col), F.lit(0)))

In [0]:
# Average velocity stats by fraud label
velocity_stats = df_clean.groupBy("fraud_label") \
                         .agg(
                             F.round(F.mean("txn_count_1h"), 4)
                              .alias("avg_txn_count_1h"),
                             F.round(F.mean("txn_count_1d"), 4)
                              .alias("avg_txn_count_1d"),
                             F.round(F.mean("txn_count_7d"), 4)
                              .alias("avg_txn_count_7d"),
                             F.round(F.mean("amount_sum_1h"), 4)
                              .alias("avg_amount_sum_1h"),
                             F.round(F.mean("amount_sum_1d"), 4)
                              .alias("avg_amount_sum_1d"),
                             F.round(F.mean("amount_sum_7d"), 4)
                              .alias("avg_amount_sum_7d")
                         ) \
                         .orderBy("fraud_label")

velocity_stats.show()

In [0]:
# Define high velocity thresholds based on percentiles
# First check distribution
df_clean.select(
    F.expr("percentile(txn_count_1h, 0.90)").alias("p90_1h"),
    F.expr("percentile(txn_count_1h, 0.95)").alias("p95_1h"),
    F.expr("percentile(txn_count_1h, 0.99)").alias("p99_1h"),
    F.expr("percentile(txn_count_1d, 0.90)").alias("p90_1d"),
    F.expr("percentile(txn_count_1d, 0.95)").alias("p95_1d"),
    F.expr("percentile(txn_count_1d, 0.99)").alias("p99_1d")
).show()

In [0]:
# Transaction count thresholds
df_clean = df_clean.withColumn("high_velocity_1h",
    F.when(F.col("txn_count_1h") >= 2, 1).otherwise(0)) \
                   .withColumn("high_velocity_1d",
    F.when(F.col("txn_count_1d") >= 6, 1).otherwise(0))

# Amount sum thresholds
# Use mean + 2 std as threshold — compute first
amount_stats = df_clean.select(
    F.mean("amount_sum_1h").alias("mean_1h"),
    F.stddev("amount_sum_1h").alias("std_1h"),
    F.mean("amount_sum_1d").alias("mean_1d"),
    F.stddev("amount_sum_1d").alias("std_1d")
).collect()[0]

threshold_1h = amount_stats["mean_1h"] + 2 * amount_stats["std_1h"]
threshold_1d = amount_stats["mean_1d"] + 2 * amount_stats["std_1d"]

print(f"Amount threshold 1h: ${threshold_1h:.2f}")
print(f"Amount threshold 1d: ${threshold_1d:.2f}")

df_clean = df_clean.withColumn("high_amount_velocity_1h",
    F.when(F.col("amount_sum_1h") > threshold_1h, 1).otherwise(0)) \
                   .withColumn("high_amount_velocity_1d",
    F.when(F.col("amount_sum_1d") > threshold_1d, 1).otherwise(0))

# Check fraud rates for each flag
for flag in ["high_velocity_1h", "high_velocity_1d",
             "high_amount_velocity_1h", "high_amount_velocity_1d"]:
    print(f"\nFraud rate by {flag}:")
    df_clean.groupBy(flag) \
            .agg(
                F.count("*").alias("total"),
                F.sum((F.col("fraud_label") == "Yes").cast("int"))
                 .alias("fraud_count")
            ) \
            .withColumn("fraud_rate",
                F.round(F.col("fraud_count") / F.col("total") * 100, 4)) \
            .orderBy(flag) \
            .show()

In [0]:
# Prepare data for plotting
velocity_data = {
    "Flag": ["Count ≥2\n(1h)", "Count ≥6\n(1d)", 
             "Amount >$98\n(1h)", "Amount >$413\n(1d)"],
    "Not_Flagged": [0.1225, 0.1104, 0.1111, 0.1015],
    "Flagged": [0.198, 0.2899, 0.5662, 0.7632]
}

df_vel_flags = pd.DataFrame(velocity_data)
df_vel = velocity_stats.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Velocity Feature Analysis (2010–2018)",
             fontsize=14, fontweight="bold")

# Plot 1: Avg transaction count by window
windows = ["1 Hour", "1 Day", "7 Days"]
fraud_counts = [
    float(df_vel[df_vel["fraud_label"]=="Yes"]["avg_txn_count_1h"]),
    float(df_vel[df_vel["fraud_label"]=="Yes"]["avg_txn_count_1d"]),
    float(df_vel[df_vel["fraud_label"]=="Yes"]["avg_txn_count_7d"])
]
non_fraud_counts = [
    float(df_vel[df_vel["fraud_label"]=="No"]["avg_txn_count_1h"]),
    float(df_vel[df_vel["fraud_label"]=="No"]["avg_txn_count_1d"]),
    float(df_vel[df_vel["fraud_label"]=="No"]["avg_txn_count_7d"])
]

x = np.arange(len(windows))
width = 0.35
axes[0].bar(x - width/2, non_fraud_counts, width,
            label="Non-Fraud", color="steelblue", alpha=0.7)
axes[0].bar(x + width/2, fraud_counts, width,
            label="Fraud", color="crimson", alpha=0.7)
axes[0].set_title("Avg Transaction Count by Window")
axes[0].set_xticks(x)
axes[0].set_xticklabels(windows)
axes[0].set_ylabel("Avg Transaction Count")
axes[0].legend()
for i, (nf, f) in enumerate(zip(non_fraud_counts, fraud_counts)):
    axes[0].text(i - width/2, nf, f'{nf:.2f}', 
                 ha="center", va="bottom", fontsize=8)
    axes[0].text(i + width/2, f, f'{f:.2f}',
                 ha="center", va="bottom", fontsize=8)

# Plot 2: Avg amount sum by window
fraud_amounts = [
    float(df_vel[df_vel["fraud_label"]=="Yes"]["avg_amount_sum_1h"]),
    float(df_vel[df_vel["fraud_label"]=="Yes"]["avg_amount_sum_1d"]),
    float(df_vel[df_vel["fraud_label"]=="Yes"]["avg_amount_sum_7d"])
]
non_fraud_amounts = [
    float(df_vel[df_vel["fraud_label"]=="No"]["avg_amount_sum_1h"]),
    float(df_vel[df_vel["fraud_label"]=="No"]["avg_amount_sum_1d"]),
    float(df_vel[df_vel["fraud_label"]=="No"]["avg_amount_sum_7d"])
]

axes[1].bar(x - width/2, non_fraud_amounts, width,
            label="Non-Fraud", color="steelblue", alpha=0.7)
axes[1].bar(x + width/2, fraud_amounts, width,
            label="Fraud", color="crimson", alpha=0.7)
axes[1].set_title("Avg Amount Sum by Window")
axes[1].set_xticks(x)
axes[1].set_xticklabels(windows)
axes[1].set_ylabel("Avg Amount Sum ($)")
axes[1].legend()
for i, (nf, f) in enumerate(zip(non_fraud_amounts, fraud_amounts)):
    axes[1].text(i - width/2, nf, f'${nf:.0f}',
                 ha="center", va="bottom", fontsize=8)
    axes[1].text(i + width/2, f, f'${f:.0f}',
                 ha="center", va="bottom", fontsize=8)

# Plot 3: Fraud rate by velocity flag
x2 = np.arange(len(df_vel_flags))
axes[2].bar(x2 - width/2, df_vel_flags["Not_Flagged"], width,
            label="Not Flagged", color="steelblue", alpha=0.7)
axes[2].bar(x2 + width/2, df_vel_flags["Flagged"], width,
            label="Flagged", color="crimson", alpha=0.7)
axes[2].set_title("Fraud Rate: Flagged vs Not Flagged")
axes[2].set_xticks(x2)
axes[2].set_xticklabels(df_vel_flags["Flag"], fontsize=9)
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].axhline(y=0.1263, color="black", linestyle="--",
                linewidth=1, label="Dataset Average (0.1263%)")
axes[2].legend(fontsize=8)
for i, row in df_vel_flags.iterrows():
    axes[2].text(i - width/2, row["Not_Flagged"],
                 f'{row["Not_Flagged"]}%',
                 ha="center", va="bottom", fontsize=7)
    axes[2].text(i + width/2, row["Flagged"],
                 f'{row["Flagged"]}%',
                 ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.show()

In [0]:
# Fraud cases by year - post Italy exclusion
yearly_fraud = df_clean.groupBy(F.year("date").alias("year")) \
                       .agg(
                           F.count("*").alias("total_txns"),
                           F.sum((F.col("fraud_label") == "Yes").cast("int"))
                            .alias("fraud_cases"),
                           F.round(F.col("fraud_cases") / F.col("total_txns") * 100, 4)
                            .alias("fraud_rate")
                       ) \
                       .orderBy("year")

# Add cumulative fraud column
from pyspark.sql import Window

w_cumulative = Window.orderBy("year").rowsBetween(
    Window.unboundedPreceding, Window.currentRow)

yearly_fraud = yearly_fraud \
    .withColumn("cumulative_fraud",
        F.sum("fraud_cases").over(w_cumulative))

yearly_fraud.show()

In [0]:
# 1. Check 2019 fraud distribution
fraud_2019 = df_geo.filter(F.year("date") == 2019) \
                   .filter(
                       F.col("merchant_state").isNull() |
                       (F.col("merchant_state") != "Italy")
                   ) \
                   .agg(
                       F.count("*").alias("total"),
                       F.sum((F.col("fraud_label") == "Yes").cast("int"))
                        .alias("fraud_cases")
                   ) \
                   .withColumn("fraud_rate",
                       F.round(F.col("fraud_cases") / F.col("total") * 100, 4))

fraud_2019.show()

# 2. Check where Italy fraud was concentrated by year
italy_by_year = df_geo.filter(F.col("merchant_state") == "Italy") \
                      .groupBy(F.year("date").alias("year")) \
                      .agg(
                          F.count("*").alias("total"),
                          F.sum((F.col("fraud_label") == "Yes").cast("int"))
                           .alias("fraud_cases")
                      ) \
                      .orderBy("year")

italy_by_year.show()